In [1]:
import subprocess
import yaml
import os
import sys
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import tempfile

import threading
source_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(source_path)

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import itertools
import random
import numpy as np
import json
import shutil

In [ ]:
model_list = ['MLPClassifier1','MLPClassifier2']
best_result = [np.inf, np.inf]
all_results = []

type_of_search = 'single_search'  # 'grid_search', 'single_search'
pretrain_modality = 'contrastive'  # 'standard', 'contrastive
train_modality = 'fine_tune' # 'fine_tune','progressive','from_scratch'
selected_model = 'DeiT-Tiny' #'DeiT-Tiny'  # Example model, can be changed
type_of_model = 'Transformer'  # 'CNN', 'Transformer'
huggingface = True  # Set to True if using Hugging Face models
load_data_from = 'pre-processed' #'zarr', 'folder'

selected_classifier = 'MLPClassifier1' #'logreg'
pretrained = False if train_modality == 'from_scratch' else True
use_external_validation = True  # Set to True if using an external validation set
loss_criterion = 'NTXentLoss' if pretrain_modality=='contrastive' else 'CrossEntropyLoss'
val_percentage = 0.2 if pretrain_modality == 'contrastive' else 1.0
use_amp = True

contrastive_mode = True if pretrain_modality == 'contrastive' else False
use_external_validation = True if pretrain_modality == 'contrastive' else use_external_validation

if pretrain_modality == 'contrastive':
    output_dir = source_path+f'\\outputs\\online_deep_feature_extraction\\{selected_model}\\{pretrain_modality}'
else:
    output_dir = source_path+f'\\outputs\\online_deep_feature_extraction\\{selected_model}\\{train_modality}'
file_IO.access_or_create_dir(output_dir)
script_name = source_path+"/scripts/fine-tuning.py"
kind = 'patches_224'  # Example kind, can be changed body, contrastive, patches_224
if pretrain_modality == 'contrastive':
    input_preprocessed=file_IO.load_preprocessed_files(kind, mode='contrastive')
    val_filename = file_IO.load_preprocessed_files(kind, mode='train')
else:
    input_preprocessed=file_IO.load_preprocessed_files(kind, mode='train')
    val_filename = file_IO.load_preprocessed_files(kind, 'val')

model_mode = 'truncated'  # 'truncation', 'full', 'truncated'
truncation='remove head'
custom_transform = False  # Set to True for custom transforms
transform_mode = 'resize'  # 'train', 'val', 'test', 'resize'
use_augmentation = False  # Set to True for data augmentation
n_sub_patches=-1

saved='old-laptop'  # 'new', 'old-laptop', 'new-laptop'
train_df = pd.read_csv(source_path+f'\\outputs\\preprocessed_data\\{input_preprocessed}')
if train_df['file_name'][0].startswith('C'):
    saved = 'new-laptop'

# preparing layer names

In [ ]:
transform = u_transforms.get_transform(selected_model, use_patches=True, custom=custom_transform, mode=transform_mode)
backbone = model_utils.get_model(name=selected_model, mode=model_mode, pretrained=pretrained, truncation=truncation, 
                                 contrastive=contrastive_mode)
out=model_utils.test_output(224,transform, backbone,huggingface=huggingface)
in_features = out.shape[1]
print(f"Model {selected_model} output features: {in_features}")

`torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True`. Falling back to eager attention. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


Model DeiT-Tiny output features: 192


In [5]:
if contrastive_mode:
    model = backbone
else:
    classificaton_head = model_utils.get_classification_head(selected_classifier,in_features)
    model = model_utils.JoinedModels(backbone, classificaton_head)
    #https://chatgpt.com/share/687124c2-47a8-8010-a4b0-5124a0bf5ecb
#print(model)
all_param_names,backbone_param_names,classifier_param_names = model_utils.get_param_names(model, pretrain_modality)
print(f"Total parameters: {len(all_param_names)}")
print(f"Backbone parameters: {len(backbone_param_names)}")
print(f"Classifier parameters: {len(classifier_param_names)}")
print(f"Backbone parameters: {backbone_param_names}")

Total parameters: 200
Backbone parameters: 0
Classifier parameters: 0
Backbone parameters: []


# launch fine-tuning

In [ ]:
clear_directory = False  # Set to True to clear the directory before saving checkpoints
total_epochs = 150  # Total number of epochs for fine-tuning
patience=150
steps=training_utils.get_progressive_training_steps(selected_model,False) #rewrite the function to take a string instead of load_contrastive=True/False
param_grid=training_utils.get_search_params(type_of_search=type_of_search, type_of_training=train_modality, type_of_model=type_of_model)

keys, values = zip(*param_grid.items())
all_combos = list(itertools.product(*values))

# Shuffle combinations
if type_of_search == 'random_search':
    random.shuffle(all_combos) 
    # Pick N random samples (e.g., 5)
    N = 30 if 30 < len(all_combos) else len(all_combos)
    experiments = all_combos[:N]
else:
    # For grid search, use all combinations
    experiments = all_combos[:]


save_common=output_dir
prev_log = os.path.join(save_common, f'{type_of_search}_results.csv')
if os.path.exists(prev_log):
    prev_results = pd.read_csv(prev_log)
    print(prev_results['best_val_loss'].min())
    print(prev_results.iloc[prev_results['best_val_loss'].idxmin()])
    #display(prev_results.iloc[prev_results['best_val_loss'].idxmin()])
    #print(prev_results['id'])
    for model in model_list:

        model_results = prev_results[prev_results['head_model'] == model]
        if not model_results.empty:
            best_result_temp = model_results['best_val_loss'].min()
            print(f"Best result for {model}: {best_result_temp}")
            best_result[model_list.index(model)]= best_result_temp
        else:
            print(f"No results found for model: {model}")
    last_index = prev_results['id'].max() if not prev_results.empty else 0
else:
    prev_results = None
    last_index = None
print(best_result)
print(f"Last index: {last_index}")
print(type_of_search)

[inf, inf]
Last index: None
single_search


In [ ]:
start = last_index + 1 if last_index is not None else 0
start = 0
for i, combo in enumerate(experiments[start:]):
    experiment_dict = dict(zip(keys, combo))
    experiment_dict['id'] = i+start
    model_name = experiment_dict['head_model']
    save_path = output_dir
    file_IO.access_or_create_dir(save_path)
    checkpoint_path=save_path+'\\checkpoints'
    file_IO.access_or_create_dir(checkpoint_path)
    if clear_directory==True:
        print(f'Clearing directory: {checkpoint_path}')
        file_IO.clear_folder(checkpoint_path)

    log_grad_norm = experiment_dict['log_grad_norm']
    optim_config = training_utils.set_search_params(experiment_dict,train_modality,total_epochs,steps,backbone_param_names,classifier_param_names)
    nn_parameters={
        'dropout': experiment_dict['dropout'],
        'n_neurons': experiment_dict['n_neurons'],
        'activation': experiment_dict.get('activation', 'relu'),  # Default to 'relu' if not specified
        'with_input_norm': experiment_dict.get('with_input_norm', None),  # Default to True if not specified
    }

    args = script_launching.DotDict(
        N_max=-1,
        patches=True,
        input_filename=input_preprocessed,
        val_filename=val_filename if use_external_validation else None,
        huggingface=huggingface,
        pooling=False,  # if true in transformer models use pooling, if false only the cls token
        custom_transform=custom_transform,  # custom transform for the dataset
        transform_mode=transform_mode,  # 'train', 'val', 'test'
        save_h5=False,
        selected_model=selected_model,  # googlenet, alexnet
        selected_classifier=selected_classifier,  # 'logreg', 'svm', 'rf', 'gbc', 'mlp', 'dt'
        truncation=truncation,
        running='new-laptop',
        saved=saved,
        model_mode=model_mode,  # 'truncation
        batch_size=64,
        select_cls=False,
        num_workers=4,
        pin_memory=True,
        show_image=False,
        checkpoint_path = checkpoint_path+"\\checkpoint.pt",
        save_path = save_path,
        total_epochs = total_epochs,
        log_grad_norm = log_grad_norm,
        use_profiler = False,
        run_epochs = total_epochs,
        plot_every = 1,
        patience = patience,
        use_amp = use_amp ,#mixed precision training,
        val_percentage= val_percentage ,#percentage of validation data used for linear evaluation,
        n_splits = 4,
        loss_criterion = loss_criterion,
        optim_config = optim_config,
        use_augmentation=use_augmentation,  # Set to True for data augmentation
        n_patches=n_sub_patches,
        contrastive_mode=contrastive_mode,  # Set to True for contrastive learning
        nn_parameters=nn_parameters,  # Dictionary with neural network parameters
        type_of_search=type_of_search,  # 'grid_search', 'single_search'
        train_modality=train_modality,  # 'fine_tune', 'progressive', 'from_scratch'
        pretrain_modality=pretrain_modality,  # 'standard', 'contrastive'
        load_data_from=load_data_from,  # 'zarr', 'folder'
    )
    file_IO.save_args(args,checkpoint_path)  # Save the arguments to a file

    script_launching.run_experiment_threaded(args,script_name)  # Test a single run first

    with open(os.path.join(save_path, "best_model_performance_temp.json"), "r") as f:
        best_model_performance = json.load(f)

    for key in experiment_dict.keys():
        if key not in best_model_performance:
            best_model_performance[key] = experiment_dict[key]
    all_results.append(best_model_performance)

    index=model_list.index(model_name)
    if best_model_performance['best_val_loss'] < best_result[index]:
        best_checkpoint = os.path.join(checkpoint_path, "checkpoint_best.pt")
        destination = os.path.join(save_path, "checkpoint_best.pt")
        shutil.copy2(best_checkpoint, destination)
        best = os.path.join(checkpoint_path, "training_plot.png")
        destination = os.path.join(save_path, "training_plot.png")
        shutil.copy2(best, destination)
        best = os.path.join(checkpoint_path, "args.txt")
        destination = os.path.join(save_path, "args.txt")
        shutil.copy2(best, destination)
        best_result[index] = best_model_performance['best_val_loss']

    all_results_df = pd.DataFrame(all_results)
    if prev_results is not None:
        all_results_df = pd.concat([prev_results, all_results_df], ignore_index=True)
    all_results_df.to_csv(os.path.join(save_common, f'{type_of_search}_results.csv'), index=False)

Starting experiment:
[STDERR] c:\Users\andre\anaconda3\envs\GeneralPurposeML\lib\site-packages\transformers\models\vit\feature_extraction_vit.py:30: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
[STDERR]   warnings.warn(
[STDERR] `torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True`. Falling back to eager attention. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.
[STDOUT] Output shape:  torch.Size([1, 192])
[STDOUT] tensor([[-3.5920e-03, -6.7164e-01, -3.2410e-01, -3.8492e-01, -5.0019e-01,
[STDOUT]          -9.1648e-02, -7.3570e-01, -3.2149e-01,  2.5384e-01, -2.9529e-01,
[STDOUT]           2.7650e-01, -9.5886e-02,  2.7223e-01, -7.8379e-01,  5.5774e-01,
[STDOUT]           1.3338e-01,  3.6457e-01, -6.6364e-01, -1.5967e-01, -2.7088e-01,
[STDOUT]          -4.2030e-01,  8.2880e+00, -2.5738e-01, -8.8083

# functions

## reload

In [2]:
def reload_modules():
    import importlib
    import utils.data_loading as data_loading
    import utils.visualization as visualization
    import utils.dataframes as dataframes
    import utils.utils_transforms as u_transforms
    import utils.training_utils as training_utils
    import utils.model_utils as model_utils
    import utils.file_IO as file_IO
    import utils.vit_rollout_mod as vit_rollout_mod
    import  utils.script_launching as script_launching
    
    importlib.reload(file_IO)
    importlib.reload(data_loading)
    importlib.reload(visualization)
    importlib.reload(dataframes)
    importlib.reload(u_transforms)
    importlib.reload(model_utils)
    importlib.reload(training_utils)
    importlib.reload(vit_rollout_mod)
    importlib.reload(script_launching)

    return data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod, script_launching
data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod, script_launching = reload_modules()